# Mango Ripeness Classifier - TensorFlow & TFLite Training
This notebook trains a **MobileNetV2** transfer learning model for 4 mango ripeness classes:
1. **Unripe**
2. **Partially Ripe**
3. **Fully Ripe**
4. **Overripe**

It evaluates accuracy and exports `mango_classifier.tflite` ready for your Flutter app.

### Step 1: Upload and Unzip Dataset
Upload your `Dataset.zip` file to Colab (drag-and-drop into the left sidebar files pane), then run the cell below.

In [ ]:
import zipfile
import os

# Unzip dataset if uploaded as zip
if os.path.exists('Dataset.zip'):
    with zipfile.ZipFile('Dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print('Dataset unzipped successfully!')
else:
    print('Dataset.zip not found. Ensure Dataset folder is present in current directory.')

### Step 2: Set Up TensorFlow & Prepare Data

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import numpy as np

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['Unripe', 'Partially Ripe', 'Fully Ripe', 'Overripe']

TRAIN_DIR = 'Dataset/Train'
TEST_DIR = 'Dataset/Test'

# Load datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Validation split (20% of train)
val_size = int(0.2 * len(train_ds))
val_ds = train_ds.take(val_size)
train_ds = train_ds.skip(val_size)

# Cache and prefetch for GPU performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print(f"Classes: {CLASS_NAMES}")

### Step 3: Build MobileNetV2 Architecture

In [ ]:
# Augmentation layers to prevent overfitting
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="input_image")
x = data_augmentation(inputs)
x = layers.Rescaling(1.0 / 255.0)(x)  # Matches Flutter input normalization
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation='softmax', name="probabilities")(x)

model = models.Model(inputs, outputs, name="mango_ripeness_detector")
model.summary()

### Step 4: Train Model (Warmup & Fine-tuning)

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# Phase 1: Train classification head
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)

# Phase 2: Fine-tune top layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stopping]
)

### Step 5: Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

### Step 6: Convert to TensorFlow Lite (.tflite) and Download

In [ ]:
# Build inference graph without augmentation
inf_inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name="input_image")
rescaled = layers.Rescaling(1.0 / 255.0)(inf_inputs)
inf_x = base_model(rescaled, training=False)
inf_x = layers.GlobalAveragePooling2D()(inf_x)
for layer in model.layers[model.layers.index(model.get_layer("global_average_pooling2d"))+1:]:
    if not isinstance(layer, layers.Dropout):
        inf_x = layer(inf_x)
inf_model = models.Model(inf_inputs, inf_x)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(inf_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('mango_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Exported mango_classifier.tflite ({len(tflite_model)/1024:.1f} KB)")

# Automatically trigger download in Google Colab
try:
    from google.colab import files
    files.download('mango_classifier.tflite')
except Exception:
    print('Download manually from Colab files pane: mango_classifier.tflite')